In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = 224
NUM_CLASSES = 14

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # مهم جدًا لتقليل RAM

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    base_model,

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
def data_generator(df, batch_size=8):
    while True:
        batch = df.sample(batch_size)

        images = []
        labels = []

        for _, row in batch.iterrows():

            img = cv2.imread(row['local_image_path'])

            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            img = img / 255.0

            images.append(img)
            labels.append(row['label'])

        # ⚠️ مهم جدًا: تجنب batch فاضي
        if len(images) == 0:
            continue

        yield np.array(images), np.array(labels)

In [ ]:
train_gen = data_generator(df, batch_size=8)

steps = len(df) // 8

history_mobilenet = model.fit(
    train_gen,
    steps_per_epoch=steps,
    epochs=10
)